In [2]:
from google.colab import drive
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/dataset"  # change if needed

Mounted at /content/drive


In [3]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224,224)),

    # 🔥 STRONG AUGMENTATION
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=15, scale=(0.8,1.2)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),

    transforms.ToTensor(),

    # 🔥 NORMALIZATION (VERY IMPORTANT)
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [4]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_data = ImageFolder(data_dir + "/train", transform=train_transforms)
val_data = ImageFolder(data_dir + "/val", transform=val_transforms)
test_data = ImageFolder(data_dir + "/test", transform=val_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

class_names = train_data.classes
print(class_names)

['Brown Planthopper', 'Rice Gall Midge', 'Rice Hispa', 'Rice Leaf Folder', 'Rice Stem Borer']


In [5]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(pretrained=True)

# 🔒 Freeze early layers
for param in model.features.parameters():
    param.requires_grad = False

# 🔥 Custom classifier
num_classes = len(class_names)

model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.classifier[1].in_features, num_classes)
)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 119MB/s] 


In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.3)

In [7]:
best_acc = 0

for epoch in range(20):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total

    print(f"Epoch {epoch+1} | Loss: {train_loss:.3f} | Val Acc: {val_acc:.2f}%")

    # 🔥 SAVE BEST MODEL
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "efficientnet_best.pth")
        print("🔥 Model saved!")

    scheduler.step()

Epoch 1 | Loss: 47.442 | Val Acc: 22.55%
🔥 Model saved!
Epoch 2 | Loss: 46.303 | Val Acc: 30.39%
🔥 Model saved!
Epoch 3 | Loss: 44.143 | Val Acc: 36.27%
🔥 Model saved!
Epoch 4 | Loss: 42.899 | Val Acc: 38.24%
🔥 Model saved!
Epoch 5 | Loss: 41.706 | Val Acc: 42.65%
🔥 Model saved!
Epoch 6 | Loss: 41.143 | Val Acc: 42.65%
Epoch 7 | Loss: 40.205 | Val Acc: 44.61%
🔥 Model saved!
Epoch 8 | Loss: 40.364 | Val Acc: 43.63%
Epoch 9 | Loss: 39.511 | Val Acc: 42.16%
Epoch 10 | Loss: 38.894 | Val Acc: 46.08%
🔥 Model saved!
Epoch 11 | Loss: 38.702 | Val Acc: 46.57%
🔥 Model saved!
Epoch 12 | Loss: 38.810 | Val Acc: 45.10%
Epoch 13 | Loss: 38.396 | Val Acc: 46.08%
Epoch 14 | Loss: 38.439 | Val Acc: 47.06%
🔥 Model saved!
Epoch 15 | Loss: 38.289 | Val Acc: 47.06%
Epoch 16 | Loss: 38.402 | Val Acc: 48.53%
🔥 Model saved!
Epoch 17 | Loss: 38.255 | Val Acc: 46.08%
Epoch 18 | Loss: 38.374 | Val Acc: 47.06%
Epoch 19 | Loss: 38.139 | Val Acc: 45.59%
Epoch 20 | Loss: 38.481 | Val Acc: 47.55%


In [9]:
# 🔓 Unfreeze last layers
for param in model.features[-2:].parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

In [10]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return 100 * correct / total

model.load_state_dict(torch.load("efficientnet_best.pth"))

test_acc = evaluate(model, test_loader)
print("🔥 Test Accuracy:", test_acc)

🔥 Test Accuracy: 66.82692307692308


In [11]:
# 🔓 Unfreeze last 3 blocks
for param in model.features[-3:].parameters():
    param.requires_grad = True

In [12]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5   # 🔥 lower LR
)

In [13]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=10, scale=(0.9,1.1)),

    transforms.ColorJitter(brightness=0.2, contrast=0.2),

    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [14]:
best_acc = 0

for epoch in range(15):
    model.train()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total
    print(f"Epoch {epoch+1} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_fixed_model.pth")

Epoch 1 | Val Acc: 49.02%
Epoch 2 | Val Acc: 50.00%
Epoch 3 | Val Acc: 53.43%
Epoch 4 | Val Acc: 53.92%
Epoch 5 | Val Acc: 55.39%
Epoch 6 | Val Acc: 57.35%
Epoch 7 | Val Acc: 57.84%
Epoch 8 | Val Acc: 57.35%
Epoch 9 | Val Acc: 59.80%
Epoch 10 | Val Acc: 60.29%
Epoch 11 | Val Acc: 62.25%
Epoch 12 | Val Acc: 61.27%
Epoch 13 | Val Acc: 62.75%
Epoch 14 | Val Acc: 63.24%
Epoch 15 | Val Acc: 63.73%


In [15]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(pretrained=True)

# 🔥 PARTIAL FREEZE (NOT FULL FREEZE)
for param in model.features[:4].parameters():
    param.requires_grad = False

for param in model.features[4:].parameters():
    param.requires_grad = True

# 🔥 BETTER CLASSIFIER
model.classifier = nn.Sequential(
    nn.Dropout(0.6),
    nn.Linear(model.classifier[1].in_features, len(class_names))
)

model = model.to(device)

In [16]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),   # 🔥 NOT filtered
    lr=1e-4
)

In [17]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [18]:
best_acc = 0

for epoch in range(20):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total

    print(f"🔥 Epoch {epoch+1} | Loss: {train_loss:.2f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "efficientnet_final.pth")
        print(f"✅ Saved best model: {best_acc:.2f}%")

🔥 Epoch 1 | Loss: 45.26 | Val Acc: 57.84%
✅ Saved best model: 57.84%
🔥 Epoch 2 | Loss: 34.38 | Val Acc: 62.25%
✅ Saved best model: 62.25%
🔥 Epoch 3 | Loss: 24.21 | Val Acc: 64.22%
✅ Saved best model: 64.22%
🔥 Epoch 4 | Loss: 16.16 | Val Acc: 69.61%
✅ Saved best model: 69.61%
🔥 Epoch 5 | Loss: 11.28 | Val Acc: 77.45%
✅ Saved best model: 77.45%
🔥 Epoch 6 | Loss: 8.47 | Val Acc: 78.43%
✅ Saved best model: 78.43%
🔥 Epoch 7 | Loss: 6.47 | Val Acc: 81.37%
✅ Saved best model: 81.37%
🔥 Epoch 8 | Loss: 6.19 | Val Acc: 80.39%
🔥 Epoch 9 | Loss: 5.59 | Val Acc: 82.84%
✅ Saved best model: 82.84%
🔥 Epoch 10 | Loss: 4.23 | Val Acc: 82.35%
🔥 Epoch 11 | Loss: 4.13 | Val Acc: 83.82%
✅ Saved best model: 83.82%
🔥 Epoch 12 | Loss: 3.35 | Val Acc: 85.78%
✅ Saved best model: 85.78%
🔥 Epoch 13 | Loss: 2.72 | Val Acc: 84.80%
🔥 Epoch 14 | Loss: 2.84 | Val Acc: 83.82%
🔥 Epoch 15 | Loss: 2.65 | Val Acc: 85.29%
🔥 Epoch 16 | Loss: 2.23 | Val Acc: 84.80%
🔥 Epoch 17 | Loss: 1.95 | Val Acc: 83.82%
🔥 Epoch 18 | Loss: 2

In [19]:
model.load_state_dict(torch.load("efficientnet_final.pth"))

test_acc = evaluate(model, test_loader)
print("🔥 Final Test Accuracy:", test_acc)

🔥 Final Test Accuracy: 91.34615384615384


In [20]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.targets),
    y=train_data.targets
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [21]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

In [22]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [23]:
for param in model.features.parameters():
    param.requires_grad = True

In [24]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-6   # 🔥 ultra low
)

In [25]:
best_acc = 0

for epoch in range(10):   # only 10 epochs
    model.train()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total

    print(f"🔥 FineTune Epoch {epoch+1} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "efficientnet_boosted.pth")

    scheduler.step()

🔥 FineTune Epoch 1 | Val Acc: 85.29%


/tmp/ipykernel_4583/413334595.py:39: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


🔥 FineTune Epoch 2 | Val Acc: 86.27%
🔥 FineTune Epoch 3 | Val Acc: 85.78%
🔥 FineTune Epoch 4 | Val Acc: 85.78%
🔥 FineTune Epoch 5 | Val Acc: 86.27%
🔥 FineTune Epoch 6 | Val Acc: 85.29%
🔥 FineTune Epoch 7 | Val Acc: 86.76%
🔥 FineTune Epoch 8 | Val Acc: 85.78%
🔥 FineTune Epoch 9 | Val Acc: 86.27%
🔥 FineTune Epoch 10 | Val Acc: 84.80%


In [26]:
model.load_state_dict(torch.load("efficientnet_boosted.pth"))

test_acc = evaluate(model, test_loader)
print("🔥 Boosted Accuracy:", test_acc)

🔥 Boosted Accuracy: 90.86538461538461
